# Monitoring Chlorophyll-*a* in Welsh waterbodies <img align="right" src="imgs/LivingWales_logo.png" width="250" alt="LivingWales logo" style="height:auto;">
* [**Sign up to the Wales Open Data Cube**](https://hub.livingwales.space/)
* **Compatibility:** Notebook currently compatible with the `WDC` environment
* **Products used:** `sen2_l2a_gcp`

***

## Background
Inland waterbodies are essential for supporting human life, both through the supply of drinking water and the support of agriculture and aquaculture.
Such waterbodies can be contaminated by outbreaks of blue-green (and other toxic) algae, causing health issues for people and animals.
For example, up to a million fish died during an [algal bloom event](https://www.abc.net.au/news/2019-01-08/second-fish-kill-in-darling-river-at-menindee/10696632) in the Darling river in late 2018 and early 2019. 
While the health of waterbodies can be monitored from the ground through sampling, satellite imagery can complement this, potentially improving the detection of large algal bloom events.
However, there needs to be a well-understood and tested way to link satellite observations to the presence of algal blooms.

### Sentinel-2 use case
Algal blooms are associated with the presence of clorophyll-*a* in waterbodies.
[Mishra and Mishra (2012)](https://doi.org/10.1016/j.rse.2011.10.016) developed the normalised difference chlorophyll index (NDCI), which serves as a qualitative indicator for the concentration of clorophyll-*a* on the surface of a waterbody.
The index requires information from a specific part of the infrared specturm, known as the 'red edge'. 
This is captured as part of Sentinel-2's 13 spectral bands, making it possible to measure the NDCI with Sentinel-2. 

## Description

In this example, we measure the NDCI for the Llangorse Lake, Wales, UK, lake, which is strongly affected by the algal bloom events.
This is combined with information about the size of the waterbody, which is used to build a helpful visualisation of how the water-level and presence of chlorophyll-*a* changes over time.
The worked example takes users through the code required to:

1. Load cloud-free Sentinel-2 images for an area of interest.
2. Compute indices to measure the presence of water and clorophyll-*a*.
3. Generate informative visualisations to identify the presence of clorophyll-*a*.

### Some caveats

* The NDCI is currently treated as an experimental index for Wales and Australia, as futher work is needed to calibrate and validate how well the index relates to the presence of clorophyll-*a*. 
* It is also important to remember that algal blooms will usually result in increased values of the NDCI, but not all NDCI increases will be from algal blooms.
For example, there may be seasonal fluctuations in the amount of clorophyll-*a* in a waterbody.
* Further validation work is required to understand how shallow water and atmospheric effects affect the NDCI, and its use in identifying high concentrations of clorophyll-*a*.

***

## Getting started

**To run this analysis**, run all the cells in the notebook, starting with the "Load packages" cell.

**After finishing the analysis**, return to the "Analysis parameters" cell, modify some values (e.g. choose a different location or time period to analyse) and re-run the analysis.
There are additional instructions on modifying the notebook at the end.

### Load packages
Load key Python packages and supporting functions for the analysis.

In [ ]:
import datacube
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("../../notebooks/wales_utils/data_cube_utilities")
from wdc_bandindices import calculate_indices
from wdc_datahandling import geom_fromdrawn, cleaning_s2, cloud_coverage, export_to_GeoTiff

sys.path.append("../isabel/wdc_utils")
from display_tools import map_extent, draw_select, calendar, rgb, cloud_threshold_slider
from plotting import rgb, display_map
from datahandling import load_ard


### Connect to the datacube
Activate the datacube database, which provides functionality for loading and displaying stored Earth observation data.

In [ ]:
dc = datacube.Datacube()

### Analysis parameters

The following cell sets the parameters, which define the area of interest and the length of time to conduct the analysis over.
The parameters are

* `latitude`: The latitude range to analyse (e.g. `(51.8782, 51.8455)`).
For reasonable loading times, make sure the range spans less than ~0.1 degrees.
* `longitude`: The longitude range to analyse (e.g. `(-3.3285, -3.2968)`).
For reasonable loading times, make sure the range spans less than ~0.1 degrees.
* `time`: The date range to analyse (e.g. `("2018-07-01", "2019-03-01")`).
For reasonable loading times, make sure the range spans less than 3 years.
Note that Sentinel-2 data is only available after July 2015.

<div class="alert alert-info">
    
**If running the notebook for the first time**, keep the default settings below.
This will demonstrate how the analysis works and provide meaningful results.
The example covers Lake Cawndilla, one of the Menindee Lakes in New South Wales.
To run the notebook for a different area, make sure Sentinel-2 data is available for the chosen area using the [DEA Explorer](https://explorer.dea.ga.gov.au/ga_s2am_ard_3).
    
</div>




### Define the area of interest

In [ ]:
# Llangors Lake
latitude = (51.9355, 51.9216)
longitude = (-3.2773, -3.2475)

# Talybont reservoir
# latitude = (51.8782, 51.8455)
# longitude = (-3.3285, -3.2968)

# Cantref Reservoir
# latitude = (51.8375, 51.8268)
# longitude = (-3.4675, -3.4511)

# # Ystwyth
# latitude = (52.4156, 52.3920)
# longitude = (-4.1091, -4.0812)


## View the selected location
The next cell will display the selected area on an interactive map.
Feel free to zoom in and out to get a better understanding of the area you'll be analysing.
Clicking on any point of the map will reveal the latitude and longitude coordinates of that point.

In [ ]:
display_map(x=longitude, y=latitude)

## Load and view Sentinel 2 data

The first step in the analysis is to load Sentinel-2 data for the specified area of interest and time range.
This uses the pre-defined `dc.load` utility function.

**Please be patient**.
The data may take a few minutes to load and progress will be indicated by text output.
The load is complete when the cell status goes from `[*]` to `[number]`.

In [ ]:
dc = datacube.Datacube()

ds_s2 = dc.load(product= 'sen2_l2a_gcp',
                  x= longitude,
                  y= latitude,
                  measurements= ['blue', 'green', 'red','veg5','nir','swir1','scl'],
                  time= ("2018-01-01", "2024-11-29"),
                  output_crs= 'epsg:27700',
                  resolution= (-10,10),
                  dask_chunks={"y" : 2048, "x" : 2048})



Now we can examine the data by printing it in the next cell.
The `Dimensions` argument revels the number of time steps in the data set, as well as the number of pixels in the `x` (longitude) and `y` (latitude) dimensions.

In [ ]:
ds_s2

### Plot example timestep in true colour

To visualise the data, use the pre-loaded `rgb` utility function to plot a true colour image for a given time-step. 

The settings below will display images for two time steps near the start and end of the timeseries.
White areas indicate where clouds or other invalid pixels in the image have been masked.
What are the key differences between the two images?

Feel free to experiement with the values for the `initial_timestep` and `final_timestep` variables; re-run the cell to plot the images for the new timesteps.
The values for the timesteps can be `0` to `n_time - 1` where `n_time` is the number of timesteps (see the `time` listing under the `Dimensions` category in the dataset print-out above).

**Note:** if the location and time are changed, you may need to change the `intial_timestep` and `final_timestep` parameters to view images at similar times of year.

In [ ]:
# Let's clean the Sentinel-2 dataset (i.e., cloud masking and reflectance normalisation)
dataset_clean = cleaning_s2(ds_s2)

In [ ]:
# # Visualise clean dataset
# print("Plotting ...")
# print("(Please wait until images appear. This may take a few seconds to minutes depending on your period of interest.)")

# dataset_clean.nir.plot(col="time", col_wrap=10);

In [ ]:
cloud_max_threshold = cloud_threshold_slider()
cloud_max_threshold

In [ ]:
# Calculating the cloud coverage (%) for each date
cloud_percentage = cloud_coverage(dataset_clean)

# Dropping dates where cloud percentage greater than cloud maximum threshold
cloud_mask = cloud_percentage<=cloud_max_threshold.value
cloud_mask = cloud_mask.compute()
dataset_2use = dataset_clean.where(cloud_mask, drop=True)

In [ ]:
if len(dataset_2use.time) == 0:
    print("No valid images found. Try increasing the maximum cloud cover, using a larger area or different dates")
else:
    print("Plotting ...")
    print("(Please wait until images appear. This may take a few seconds)")
    dataset_2use.nir.plot(col='time', col_wrap=6);

In [ ]:
# Set the timesteps to visualise
initial_timestep = 0
final_timestep = 100
# Generate RGB plots at each timestep
#rgb(dataset_2use,col='time',col_wrap=4, percentile_stretch=(0.3,0.95))
rgb(dataset_2use,index=[1,2,15,16,17,18,27,-1],percentile_stretch=(0.3,0.95))  #[75,76,77,84,85,86,87,88,93,94,95]


## Compute band indices
This study measures the presence of water through the modified normalised difference water index (MNDWI) and clorophyll-*a* through the normalised difference clorophyll index (NDCI).

MNDWI is calculated from the green and shortwave infrared (SWIR) bands to identify water.
The formula is

$$
\begin{aligned}
\text{MNDWI} = \frac{\text{Green} - \text{SWIR}}{\text{Green} + \text{SWIR}}.
\end{aligned}
$$

When interpreting this index, values greater than 0 indicate water.

NDCI is calculated from the red edge 1 and red bands to identify water.
The formula is

$$
\begin{aligned}
\text{NDCI} = \frac{\text{Red edge 1} - \text{Red}}{\text{Red edge 1} + \text{Red}}.
\end{aligned}
$$

When interpreting this index, high values indicate the presence of clorophyll-*a*.

In [ ]:
# Calculate NDCI and MNDWI and add it to the loaded data set
dataset_2use = calculate_indices(dataset_2use, index=['MNDWI', 'NDCI'], platform="SENTINEL_2",
                                normalise=True)

The MNDWI and NDCI values should now appear as data variables, along with the loaded measurements, in the `ds_s2` data set.
Check this by printing the data set below:

In [ ]:
dataset_2use

## Build summary plot
To get an understanding of how the waterbody has changed over time, the following section builds a plot that uses the MNDWI to measure the rough area of the waterbody, along with the NDCI to track how the concentration of clorophyll-*a* is changing over time.
This could be used to quickly assess the status of a given waterbody.

### Set up constants
The number of pixels classified as water (MNDWI > 0) can be used as a proxy for the area of the waterbody if the pixel area is known. 
Run the following cell to generate the necessary constants for performing this conversion.

In [ ]:
# Constants for calculating waterbody area
pixel_length = dataset_2use.odc.geobox.resolution.x  # in metres
m_per_km = 1000  # conversion from metres to kilometres

area_per_pixel = pixel_length**2 / m_per_km**2

### Compute the total water area
The next cell starts by filtering the data set to only keep the pixels that are classified as water. 
It then calculates the water area by counting all of the MNDWI pixels in the filtered data set, calculating a rolling median (this helps smooth the results to account for variation from cloud-masking), then multiplies this median count by the area-per-pixel.

In [ ]:
# Filter the data to contain only pixels classified as water
ds_s2_waterarea = dataset_2use.where(dataset_2use.MNDWI > 0.0)

# Calculate the total water area (in km^2)
waterarea = (
    ds_s2_waterarea.MNDWI.count(dim=["x", "y"])
    .rolling(time=3, center=True, min_periods=1)
    .median(skipna=True)
    * area_per_pixel
)

# Plot the resulting water area through time
lake_area_fig, axes = plt.subplots(1, 1, figsize=(12, 4))
waterarea.plot()
axes.set_xlabel("Date")
axes.set_ylabel("Waterbody area (km$^2$)")
#axes.set_title("Cantref area")
plt.show()

In [ ]:
lake_area_fig.savefig("./storymap_figs_tables/cantref_area_2018_2024.png", dpi=300, bbox_inches='tight')

### Compute the average NDCI
The next cell computes the average NDCI for each time step using the filtered data set.
This means that we're only tracking the NDCI in waterbody areas, and not on any land.
To make the summary plot, we calculate NDCI across all pixels; this allows us to track overall changes in NDCI, but doesn't tell us where the increase occured within the waterbody (this is covered in the next section).

In [ ]:
# Calculate the average NDCI
average_ndci = ds_s2_waterarea.NDCI.mean(dim=["x", "y"], skipna=True)

# Plot average NDCI through time
ndci_fig, axes = plt.subplots(1, 1, figsize=(12, 4))
average_ndci.plot()
axes.set_xlabel("Date")
axes.set_ylabel("Average NDCI")
plt.show()

In [ ]:
ndci_fig.savefig("./storymap_figs_tables/lake_cantref_ndci_2018_2024.png", dpi=300, bbox_inches='tight')

### Combine the data to build the summary plot
The cell below combines the total water area and average NDCI time series we generated above into a single summary plot. 
Notice that Python can be used to build highly-customised plots.
If you're interested, take some time to understand how the plot is built.
Otherwise, run the cell to build the plot.

In [ ]:
# Set up the figure
ndci_sum_fig, axes = plt.subplots(1, 1, figsize=(12, 4))

# Set the colour map to use and the normalisation. NDCI is plotted on a scale
# of -0.1 to 0.3, so the colour map is normalised to these values
min_ndci_scale = -0.4
max_ndci_scale = 0.4
cmap = plt.get_cmap("viridis")
normal = plt.Normalize(vmin=min_ndci_scale, vmax=max_ndci_scale)

# Store the dates from the data set as numbers for ease of plotting
dates = matplotlib.dates.date2num(ds_s2_waterarea.time.values)

# Add the basic plot to the figure
# This is just a line showing the area of the waterbody over time
axes.plot_date(x=dates, y=waterarea, color="black", linestyle="-", marker="")

# Fill in the plot by looping over the possible threshold values and filling
# the areas that are greater than the threshold
color_vals = np.linspace(0, 1, 100)
threshold_vals = np.linspace(min_ndci_scale, max_ndci_scale, 100)
for ii, thresh in enumerate(threshold_vals):
    im = axes.fill_between(dates,
                           0,
                           waterarea,
                           where=(average_ndci >= thresh),
                           norm=normal,
                           facecolor=cmap(color_vals[ii]),
                           alpha=1)

# Add the colour bar to the plot
cax, _ = matplotlib.colorbar.make_axes(axes)
cb2 = matplotlib.colorbar.ColorbarBase(cax, cmap=cmap, norm=normal)
cb2.set_label("NDCI")

# Add titles and labels to the plot
axes.set_xlabel("Date")
axes.set_ylabel("Waterbody area (km$^2$)")
#axes.set_title("Cantref NDCI 2018-2024")
plt.show()

What does the plot reveal about the waterbody? 
Are there periods that show high NDCI values?

In [ ]:
ndci_sum_fig.savefig("./storymap_figs_tables/lake_cantref_ndci_summary_2018_2024.png", dpi=300, bbox_inches='tight')

## Compare spatial NDCI at two different dates
While the summary plot is useful at a glance it can be interesting to see the full spatial picture at times where the NDCI is low vs. high.
The code below defines two useful functions: `closest_date` will find the date in a list of dates closest to any given date; `date_index` will return the position of a particular date in a list of dates. 
These functions are useful for selecting images to compare. 

In [ ]:
def closest_date(list_of_dates, desired_date):
    return min(list_of_dates,
               key=lambda x: abs(x - np.datetime64(desired_date)))

def date_index(list_of_dates, known_date):
    return (np.where(list_of_dates == known_date)[0][0])

Run the cell below to set two dates to compare.
Feel free to change the dates to look at the NDCI of the waterbody at different times.

In [ ]:
# Set the dates to view
date_1 = "2018-08-08"
date_2 = "2018-09-27"

# Compute the closest date available from the data set
closest_date_1 = closest_date(dataset_2use.time.values, date_1)
closest_date_2 = closest_date(dataset_2use.time.values, date_2)

# Make an xarray containing the closest dates, which is used to select
# the dates from the data set
time_xr = xr.DataArray([closest_date_1, closest_date_2], dims=["time"])


# Plot the NDCI values for pixels classified as water for the two dates.
ndci_plot = ds_s2_waterarea.NDCI.sel(time=time_xr).plot.imshow(col="time",
                                                   cmap=cmap,
                                                   vmin=min_ndci_scale,
                                                   vmax=max_ndci_scale,
                                                   col_wrap=2,
                                                   robust=True,
                                                   figsize=(12,6))


# Access the underlying matplotlib figure
for ax in ndci_plot.fig.axes:
    ax.set_title(ax.get_title(), fontsize=10)  # Adjust title fonts if needed



plt.show()

In [ ]:
# Set the date to view
date_1 = "2018-09-27"

# Compute the closest date available from the data set
closest_date_1 = closest_date(dataset_2use.time.values, date_1)

# Select the data for the closest date
ndci_single_date = ds_s2_waterarea.NDCI.sel(time=closest_date_1)

# Squeeze the data to remove the singleton time dimension
ndci_single_date = ndci_single_date.squeeze()

# Plot the NDCI values for pixels classified as water for the selected date
plt.figure(figsize=(12, 8))
ndci_plot = ndci_single_date.plot.imshow(
    cmap=cmap,
    vmin=min_ndci_scale,
    vmax=max_ndci_scale,
    robust=True
)

# Set the title
plt.title(f"NDCI on {str(closest_date_1)[:10]}", fontsize=12)

# Save the figure
save_path = "./storymap_figs_tables/cantref_ndci_plot_2018-09-27.png"
plt.savefig(save_path, dpi=300, bbox_inches="tight")

print(f"Plot saved to {save_path}")

plt.show()



In [ ]:
ndci_plot.fig.savefig("./storymap_figs_tables/cantref_ndci_plot_2020.png", dpi=300, bbox_inches='tight')

Run the cell below to see the true-colour images for the same dates

In [ ]:
# Compute the index of the closest dates
closest_date_1_idx = date_index(dataset_2use.time.values, closest_date_1)
closest_date_2_idx = date_index(dataset_2use.time.values, closest_date_2)

# Make the true colour plots for the closest dates
rgb(dataset_2use, index=[closest_date_1_idx, closest_date_2_idx])
plt.savefig("./storymap_figs_tables/cantref_true_color_plot_2020.png", dpi=300, bbox_inches='tight')

## Drawing conclusions
Here are some questions to think about:

* How well do increased NDCI values match with what you see in the true-colour images?
* When would it be more helpful to use the summary plot vs. the time comparison plot and vice-versa?
* What does the water look like in the true-colour image when the NDCI is low-to-medium vs. high?
* What other information might you seek if you detected an abnormally high NDCI value?

## Next steps

When you are done, return to the "Analysis parameters" section, modify some values (e.g. `latitude`, `longitude` or `time`) and rerun the analysis.
You can use the interactive map in the "View the selected location" section to find new central latitude and longitude values by panning and zooming, and then clicking on the area you wish to extract location values for.
You can also use Google maps to search for a location you know, then return the latitude and longitude values by clicking the map.

If you want to look at another lakes (e.g., Vyrnwy), try the following coordinates over the same time period as the original example:

```
latitude = (52.8076,52.7497)
longitude = (-3.5699,-3.4169)
```

---

## Additional information

**License:** The code in this notebook is licensed under the [Apache License, Version 2.0](https://www.apache.org/licenses/LICENSE-2.0). 
Digital Earth Australia data is licensed under the [Creative Commons by Attribution 4.0](https://creativecommons.org/licenses/by/4.0/) license.

**Contact:** If you need assistance, please post a question on the [Open Data Cube Discord chat](https://discord.com/invite/4hhBQVas5U) or on the [GIS Stack Exchange](https://gis.stackexchange.com/questions/ask?tags=open-data-cube) using the `open-data-cube` tag (you can view previously asked questions [here](https://gis.stackexchange.com/questions/tagged/open-data-cube)).
If you would like to report an issue with this notebook, you can file one on [GitHub](https://github.com/GeoscienceAustralia/dea-notebooks).

**Last modified:** July 2024

**Compatible datacube version:** 

In [ ]:
print(datacube.__version__)

## Tags
<!-- Browse all available tags on the DEA User Guide's [Tags Index](https://knowledge.dea.ga.gov.au/genindex/) -->